# Step 4A: Momentum Strategy Signal Construction

This notebook demonstrates the signal generation and trade extraction steps of the **Moving Average Crossover Momentum Strategy** for the Nifty 50 Index. It integrates the modular code from `src/indicators.py`, `src/validators.py`, and `src/momentum.py`.

In [ ]:
import sys
import os
import pandas as pd
import logging

# Ensure project root is in python path
sys.path.append(os.path.abspath('..'))

from src.momentum import MomentumSignalGenerator

# Setup logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(name)s: %(message)s')

## 1. Load Processed Clean Dataset

We load the clean Nifty 50 dataset. Let's inspect its columns and data types.

In [ ]:
clean_data_path = "../data/processed/nifty50_clean.parquet"
df = pd.read_parquet(clean_data_path)
print(f"Loaded dataset with {len(df)} rows and columns: {list(df.columns)}")
df.head()

## 2. Generate Momentum Signals

We run the signal generator using standard parameters (Short Window = 20, Long Window = 100). The generator will:
- Calculate `SMA_20` and `SMA_100`.
- Generate `Raw_Signal` (1 if Short SMA > Long SMA, else 0).
- Generate `Execution_Signal` (shifted by 1 trading day to prevent look-ahead bias).
- Generate `Position` (equal to Execution_Signal).
- Run structural, parameters, and look-ahead validation checks.

In [ ]:
generator = MomentumSignalGenerator(short_window=20, long_window=100)
signals_df = generator.generate_signals(df, close_col="Close")
signals_df.tail()

## 3. Look-Ahead Shift Verification

Let's print some rows where signals transition to confirm that `Execution_Signal` is delayed by exactly 1 day relative to `Raw_Signal`. This delay is necessary because we can only execute trades on tomorrow's market based on today's closing information.

In [ ]:
# Find rows where Raw_Signal changed
signal_diff = signals_df["Raw_Signal"].diff().fillna(0) != 0
change_dates = signals_df[signal_diff].index

# Display 3 rows around the first signal change
if len(change_dates) > 0:
    target_date = change_dates[0]
    loc = signals_df.index.get_loc(target_date)
    display(signals_df.iloc[loc-2:loc+3][["Close", "SMA_20", "SMA_100", "Raw_Signal", "Execution_Signal", "Position"]])

## 4. Extract Trade Metrics

We identify and track each complete BUY-SELL cycle. The resulting table captures transition timestamps and holding periods.

In [ ]:
trades_df = generator.detect_trades(signals_df, position_col="Position")
print(f"Extracted {len(trades_df)} trades:")
trades_df.head(10)

## 5. Calculate Performance Statistics

We compute trade statistics such as total signals, number of trades, maximum/minimum/average holding periods, and percentage of time invested.

In [ ]:
stats = generator.compute_statistics(signals_df, trades_df)
for key, value in stats.items():
    print(f"{key.replace('_', ' '):<45} : {value:.2f}" if isinstance(value, float) else f"{key.replace('_', ' '):<45} : {value}")

## 6. Generate and Display Visualizations

We run the plotting engine to generate publication-quality figures and verify them inline.

In [ ]:
figures_dir = "../reports/figures"
generator.plot_performance(signals_df, trades_df, output_dir=figures_dir)

from IPython.display import Image, display
display(Image(filename="../reports/figures/price_sma_crossover.png"))
display(Image(filename="../reports/figures/signal_timeline.png"))
display(Image(filename="../reports/figures/position_timeline.png"))
display(Image(filename="../reports/figures/trade_durations.png"))